#what this notebook does: connect to anki via plugin running on 8765. Search for note type "Moritz Language Reactor" with a "Notes" field that contains "left-align" html tag. then do the following: search for the first "<left-align>" tag in the "Notes" field and replace it with <div style="text-align:left;">. The closing "</left-align> should be replaced with a closing </div> 

In [22]:
import requests
import json

# AnkiConnect URL
url = "http://localhost:8765"

# Function to invoke AnkiConnect
def invoke(action, **params):
	response = requests.post(url, json={"action": action, "version": 6, "params": params})
	return response.json()

# Find notes of type "Moritz Language Reactor" with "Notes" field containing "<left-align>"
query = 'note:"Moritz Language Reactor" Notes:*<left-align>*'
note_ids = invoke("findNotes", query=query)["result"]

# # Process each note
# for note_id in note_ids:
# 	# Get note info
# 	note_info = invoke("notesInfo", notes=[note_id])["result"][0]
# 	notes_field = note_info["fields"]["Notes"]["value"]
	
# 	# Replace first <left-align> with <div style="text-align:left;">
# 	notes_field = notes_field.replace("<left-align>", '<div style="text-align:left;">', 1)
# 	# Replace first </left-align> with </div>
# 	notes_field = notes_field.replace("</left-align>", '</div>', 1)
	
# 	# Update the note
# 	invoke("updateNoteFields", note={"id": note_id, "fields": {"Notes": notes_field}})

In [23]:
# Find all notes of this type
query = 'note:"Moritz Language Reactor"'
note_ids = invoke("findNotes", query=query)["result"]

# Delete Notes content if it has fewer than 20 non-whitespace symbols
min_symbols = 30

# Read in larger batches to reduce notesInfo round trips
read_batch_size = 500
note_ids_to_clear = []

for i in range(0, len(note_ids), read_batch_size):
    batch_ids = note_ids[i:i + read_batch_size]
    notes = invoke("notesInfo", notes=batch_ids)["result"]

    for note in notes:
        notes_field = note["fields"]["Notes"]["value"]
        symbol_count = len("".join(notes_field.split()))

        if symbol_count < min_symbols:
            note_ids_to_clear.append(note["noteId"])

# Write in chunks using AnkiConnect multi to reduce update round trips
write_batch_size = 200
for i in range(0, len(note_ids_to_clear), write_batch_size):
    chunk_ids = note_ids_to_clear[i:i + write_batch_size]
    actions = [
        {
            "action": "updateNoteFields",
            "version": 6,
            "params": {
                "note": {
                    "id": note_id,
                    "fields": {"Notes": ""}
                }
            }
        }
        for note_id in chunk_ids
    ]
    invoke("multi", actions=actions)

updated_count = len(note_ids_to_clear)
print(f"Updated {updated_count} notes.")


Updated 4906 notes.
